# Q2: Gallantry vs Civilian Winners (Age and Posthumous Awards)

**Hypotheses**
- **H0:** No age difference, and gallantry awards are not more often posthumous
- **H1:** Gallantry winners are younger, but gallantry awards are *not* usually posthumous
- **H3:** Gallantry winners are younger, *and* gallantry awards are usually posthumous

**Setup:** gallantry = Ashoka Chakra + Param Vir Chakra, civilian = Bharat Ratna + Padma awards.
"Usually posthumous" means more than 50%. Significance level: **α = 0.05**.

**Hypothesis:** Part A compares mean ages (two-sample t-test). Part B tests the posthumous proportion (binomial / z-test).

## Loading all cleaned files

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

civil  = pd.read_csv("../datasets/cleaned_datasets/national_awards_cleaned.csv")
ashoka = pd.read_csv("../datasets/cleaned_datasets/ashoka_chakra_cleaned.csv")
pvc    = pd.read_csv("../datasets/cleaned_datasets/param_vir_chakra_cleaned.csv")

print("Civilian:", len(civil), "| Ashoka Chakra:", len(ashoka), "| Param Vir Chakra:", len(pvc))

## Combining the two gallantry files into one table
The two files write "posthumous" differently (`True/False` and `Yes/No`), so we convert both to True/False.

In [ ]:
# Ashoka Chakra: the column is already True/False
ashoka_table = pd.DataFrame({
    "Award":      "Ashoka Chakra",
    "Posthumous": ashoka["Posthumously"],
    "Age":        ashoka["AgeAtAward"],
})

# Param Vir Chakra: converting "Yes"/"No" into True/False
pvc_table = pd.DataFrame({
    "Award":      "Param Vir Chakra",
    "Posthumous": pvc["Posthumous"] == "Yes",
    "Age":        pvc["AgeAtAward"],
})

# one table
gallantry = pd.concat([ashoka_table, pvc_table], ignore_index=True)

# dropping rows with missing ages
gallantry_age = gallantry["Age"].dropna()
civil_age     = civil["AgeAtAward"].dropna()

print("Gallantry winners:", len(gallantry), "(with age:", len(gallantry_age), ")")
print("Civilian winners with age:", len(civil_age))

## Distribution of Param Vir Chakra Recipients

In [ ]:
# Use the already loaded pvc dataset
df = pvc.copy()

# Compute the X-axis metric:
# AgeAtAward represents the age at the time of receiving the award (or age at death for posthumous awards)
df['Plot_Age'] = df['AgeAtAward']

# Set up the Seaborn and Matplotlib visualization style
plt.figure(figsize=(12, 6))

# Create a combined histogram and kernel density estimation (KDE) plot
ax = sns.histplot(
    data=df,
    x='Plot_Age',
    bins=range(int(df['Plot_Age'].min()), int(df['Plot_Age'].max()) + 2),
    kde=True,
    color='#1f77b4',
    edgecolor='black',
    alpha=0.6,
)

# Customize titles and labels
plt.title(
    'Distribution of Param Vir Chakra Recipients\n(Age at Award / Age at Death if Posthumous)',
    fontsize=14,
    fontweight='bold',
    pad=15,
)
plt.xlabel(
    'Age (Years)\n[Age at Award, or Age at Death for Posthumous Awards]',
    fontsize=12,
    labelpad=10,
)
plt.ylabel('Number of People (Frequency)', fontsize=12, labelpad=10)

# Add gridlines and layout adjustments
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Display the plot window
plt.show()

# Part A: Are gallantry winners younger than civilian winners?

### Descriptive statistics

In [ ]:
def describe(ages):
    return pd.Series({
        "Count":   len(ages),
        "Mean":    ages.mean(),
        "Median":  ages.median(),
        "Std Dev": ages.std(),
        "Q1":      ages.quantile(0.25),
        "Q3":      ages.quantile(0.75),
        "IQR":     ages.quantile(0.75) - ages.quantile(0.25),
      # interquartile range
    })

pd.DataFrame({"Gallantry": describe(gallantry_age),
              "Civilian":  describe(civil_age)}).round(2)

### Graphs for age comparison

In [ ]:
# Putting both groups in one table for a combined plot
ages = pd.concat([
    pd.DataFrame({"Age": gallantry_age, "Group": "Gallantry"}),
    pd.DataFrame({"Age": civil_age,     "Group": "Civilian"}),
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: histogram (density so the different group sizes are comparable)
sns.histplot(data=ages, x="Age", hue="Group", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Age at award: histogram")

# Right: boxplot
sns.boxplot(data=ages, x="Group", y="Age", order=["Gallantry", "Civilian"], color="lightsteelblue", ax=axes[1])
axes[1].set_title("Age at award: boxplot")

plt.tight_layout()
plt.show()

### Hypothesis test (two-sample t-test)
- **H0:** mean age of gallantry winners = mean age of civilian winners
- **H1:** mean age of gallantry winners < mean age of civilian winners (one-sided)

The civilian standard deviation is much larger than the gallantry one, so we use the **unequal-variance** version of the t-test.

In [ ]:
# Sample sizes, means, standard deviations
n1, mean1, sd1 = len(gallantry_age), gallantry_age.mean(), gallantry_age.std()
n2, mean2, sd2 = len(civil_age),     civil_age.mean(),     civil_age.std()

# Standard error of the difference in means
se = np.sqrt(sd1**2 / n1 + sd2**2 / n2)

# Test statistic
t = (mean1 - mean2) / se

# Degrees of freedom (formula for unequal variances)
df = se**4 / ((sd1**2 / n1)**2 / (n1 - 1) + (sd2**2 / n2)**2 / (n2 - 1))

# One-sided p-value (lower tail, because H1 says gallantry < civilian)
p_age = stats.t.cdf(t, df)

print("Difference in means (gallantry - civilian):", round(mean1 - mean2, 2), "years")
print("t statistic:", round(t, 2))
print("Degrees of freedom:", round(df, 1))
print("p-value:", p_age)
print()
if p_age < 0.05:
    print("p < 0.05  ->  REJECT H0: gallantry winners are significantly younger.")
else:
    print("p >= 0.05  ->  FAIL TO REJECT H0.")

In [ ]:
# 95% confidence interval for the difference in mean ages
margin = stats.t.ppf(0.975, df) * se
print("95% CI for the difference:", round((mean1 - mean2) - margin, 2), "to", round((mean1 - mean2) + margin, 2), "years")

# Quick check with scipy (should give the same t and p)
check = stats.ttest_ind(gallantry_age, civil_age, equal_var=False, alternative="less")
print("scipy check -> t:", round(check.statistic, 2), "| p:", check.pvalue)

# Part B: Are gallantry awards usually posthumous?

### Counts and percentages

In [ ]:
# Counting posthumous awards for each award type
counts = gallantry.groupby("Award")["Posthumous"].agg(Posthumous="sum", Total="count")
counts.loc["Combined"] = [counts["Posthumous"].sum(), counts["Total"].sum()]
counts["Percent posthumous"] = (counts["Posthumous"] / counts["Total"] * 100).round(1)
counts

### Graphs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: number of posthumous vs living winners
gallantry["Status"] = gallantry["Posthumous"].map({True: "Posthumous", False: "Living"})
sns.countplot(data=gallantry, x="Award", hue="Status", ax=axes[0])
axes[0].set_title("Posthumous vs living winners")
axes[0].set_ylabel("Number of winners")

# Right: percentage posthumous, with a line at 50%
sns.barplot(x=counts.index, y=counts["Percent posthumous"], ax=axes[1])
axes[1].axhline(50, color="black", linestyle="--")
axes[1].set_title("Percent posthumous (dashed line = 50%)")
axes[1].set_ylabel("Percent")
axes[1].set_xlabel("")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

### Hypothesis test (proportion)
Let p = true proportion of gallantry awards that are posthumous.
- **H0:** p = 0.5
- **H1:** p > 0.5 (usually posthumous)

If H0 is true, the number of posthumous awards follows a Binomial(n, 0.5) distribution. The p-value is the probability of seeing at least this many posthumous awards.

In [ ]:
k = gallantry["Posthumous"].sum()   # number of posthumous awards
n = len(gallantry)                  # total gallantry awards
p_hat = k / n                       # sample proportion

# Exact p-value: P(X >= k) when X ~ Binomial(n, 0.5)
p_post = stats.binom.sf(k - 1, n, 0.5)

# Same test using the normal approximation (CLT for a proportion)
z = (p_hat - 0.5) / np.sqrt(0.5 * 0.5 / n)
p_post_z = stats.norm.sf(z)

print("Posthumous:", k, "out of", n, "->", round(p_hat * 100, 1), "%")
print("Exact binomial p-value:", p_post)
print("z statistic:", round(z, 2), "| z-test p-value:", p_post_z)
print()
if p_post < 0.05:
    print("p < 0.05  ->  REJECT H0: gallantry awards are usually posthumous.")
else:
    print("p >= 0.05  ->  FAIL TO REJECT H0.")

# 95% confidence interval for p
margin_p = 1.96 * np.sqrt(p_hat * (1 - p_hat) / n)
print("95% CI for p:", round(p_hat - margin_p, 3), "to", round(p_hat + margin_p, 3))

# Final Conclusion

In [ ]:
age_reject  = p_age < 0.05
post_reject = p_post < 0.05

print("Part A (younger?):          reject H0 =", age_reject)
print("Part B (usually posthumous?): reject H0 =", post_reject)
print()

if age_reject and post_reject:
    print("DECISION: Reject H0. H3 is supported.")
elif age_reject and not post_reject:
    print("DECISION: Reject H0. H1 is supported.")
elif not age_reject and not post_reject:
    print("DECISION: Fail to reject H0.")
else:
    print("DECISION: Mixed result (not covered by H0, H1 or H3).")

### Final Conclusion of Question 2

Gallantry winners are much younger than civilian winners: the mean age is about 33.7 years (n = 77) against about 64.7 years (n = 3,561), a gap of about 31 years (95% CI: 29.0 to 32.9). The one-sided unequal-variance t-test gives t ≈ -31.6 with p ≈ 10⁻⁴⁹, far below 0.05.

82 of 107 gallantry awards (76.6%) were posthumous. The exact binomial test of H0: p = 0.5 against H1: p > 0.5 gives p ≈ 1.5 × 10⁻⁸, and the 95% CI for p is (0.69, 0.85), entirely above 0.5.

Both null hypotheses are rejected, so we **reject H0 and support H3**.
